# Cuaderno 1 — Descarga de datos

**Tesis:** Predicción del Punto de Equilibrio de Flujo de Caja en Empresas de Crecimiento  
**Autores:** Andrea Meneses · Juan Ramos  
**Universidad EAFIT · 2026**

---

## Objetivo

Construir el dataset histórico de variables financieras para las empresas del universo de estudio, descargando únicamente los datos que se van a usar.

---

## Universo de estudio: Russell 2500

La tesis estudia empresas de crecimiento de pequeña y mediana capitalización (*small y mid-cap*). El universo se define siguiendo la metodología del **Russell 2500** — el índice de referencia institucional para el segmento *small/mid-cap* del mercado accionario estadounidense (FTSE Russell, 2026).

El Russell 2500 incluye las 2.500 empresas más pequeñas del Russell 3000, excluyendo las 500 de mayor capitalización. Esto garantiza que el universo esté compuesto exclusivamente por empresas de crecimiento con dinámica de caja distinta a las grandes corporaciones consolidadas.

### Exclusiones sectoriales

Se excluyen cuatro grupos de sectores cuya dinámica financiera distorsiona el FCF operativo:

| Sector | Códigos SIC | Razón |
|---|---|---|
| Financials | 6000–6799 | Los bancos y aseguradoras crean dinero — su FCF no mide lo mismo |
| Utilities | 4900–4999 | Monopolios regulados con FCF estructuralmente positivo |
| Real Estate | 6500–6552 | El FCF depende de compraventas de propiedades, no de la operación |
| Oil & Gas | 1300–1399, 2900–2999 | El FCF depende del precio del petróleo, no de la salud del negocio |

### Exclusión de empresas extranjeras (ADRs)

Se excluyen empresas extranjeras que cotizan en NYSE/Nasdaq como ADRs pero que reportan bajo estándares contables de sus países de origen, no bajo US GAAP. Estas empresas no tienen datos disponibles en SEC EDGAR bajo el estándar US GAAP.

---

## Fuente de datos: SEC EDGAR

SEC EDGAR es la base de datos pública de la Comisión de Valores de Estados Unidos. Se usan dos tipos de formularios:
- **10-Q**: reporte trimestral (no auditado)
- **10-K**: reporte anual (auditado)

Usar ambos formularios maximiza la cobertura histórica.

---

## Variables descargadas (40)

| Grupo | Variables |
|---|---|
| Flujo de caja | FCF operativo, FCF inversor, FCF financiero, CAPEX, depreciación, stock-based compensation, emisión/recompra de acciones |
| Balance general | Caja, activos totales/corrientes, pasivos corrientes, deuda LP/CP, inventario, cuentas por cobrar/pagar, intangibles, goodwill, PPE, patrimonio, retained earnings, ingresos diferidos, impuestos pagados |
| Estado de resultados | Ingresos, costo de ventas, utilidad bruta/operativa/neta, gastos I+D, gastos operativos, gastos SGA, EBITDA proxy, intereses pagados, amortización de intangibles |
| Capital de trabajo | Acciones en circulación, cambio en inventario, cambio en cuentas por cobrar, otras inversiones, cambio en capital de trabajo |

---

## Flujo del cuaderno

```
1. Constituyentes Russell 2500 (ETF SMMD de BlackRock)
         ↓
2. Cruzar con CIKs de la SEC → universo final ~1.675 empresas
         ↓
3. Descargar 40 variables financieras solo para esas empresas
         ↓
4. Consolidar y guardar
```

> **Referencia:** FTSE Russell. (2026). *Russell US Indexes: Construction and Methodology*. LSEG.

## 1. Configuración inicial

Carga de librerías y credenciales. La SEC EDGAR requiere un `User-Agent` con nombre y correo.

Archivo `.env` en la raíz del proyecto:
```
SEC_USER_AGENT=Tu Nombre tuemail@email.com
```

In [5]:
import requests
import pandas as pd
import time
import os
import shutil
from io import StringIO
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
headers_sec = {"User-Agent": os.getenv("SEC_USER_AGENT")}

# Crear carpetas necesarias
Path("../datos/crudos/chunks").mkdir(parents=True, exist_ok=True)
Path("../datos/procesados").mkdir(parents=True, exist_ok=True)

print(f"User-Agent correcto")
print("Carpetas listas.")

User-Agent correcto
Carpetas listas.


## 2. Constituyentes del Russell 2500

Descargamos los holdings actuales del ETF **SMMD** (iShares Russell 2500 ETF) de BlackRock, que replica el índice Russell 2500. BlackRock publica estos holdings diariamente de forma pública y gratuita.

Este listado define exactamente qué empresas componen el universo de estudio.

### Filtro de tipo de activo

El CSV de holdings incluye el campo `Asset Class` para cada posición. Se filtran únicamente las posiciones clasificadas como `Equity` — acciones individuales de empresas. Este filtro excluye automáticamente cualquier ETF, fondo o instrumento derivado que pudiera aparecer como holding interno del SMMD (por ejemplo, el iShares Russell 2000 ETF que el SMMD usa para ganar exposición a una parte del índice). Filtrar por tipo de activo es más robusto que excluir tickers específicos, ya que no depende de conocer de antemano qué instrumentos no son acciones individuales.

### Limitación: survivorship bias

El SMMD refleja la composición **actual** del Russell 2500, no su composición histórica. Esto introduce un sesgo de supervivencia (*survivorship bias*) en el dataset: empresas que formaron parte del índice en años anteriores pero fueron excluidas posteriormente — por quiebra, adquisición o cambio de capitalización — no están representadas. De igual forma, empresas que ingresaron recientemente al índice tienen menor historia disponible.

Este sesgo implica que el modelo tiende a entrenarse sobre empresas que 'sobrevivieron' hasta hoy, lo que puede llevar a subestimar el riesgo real de no alcanzar el breakeven. Se reconoce como una limitación metodológica de la investigación.

In [6]:
url_smmd = (
    "https://www.ishares.com/us/products/288024/ishares-russell-2500-etf/"
    "1467271812596.ajax?fileType=csv&fileName=SMMD_holdings&dataType=fund"
)
resp = requests.get(url_smmd, headers={"User-Agent": "Mozilla/5.0"})

# Las primeras 9 líneas son metadata del fondo — saltarlas
lineas = resp.text.split("\n")
csv_limpio = "\n".join(lineas[9:])
df_russell = pd.read_csv(StringIO(csv_limpio))

# Filtrar solo acciones individuales
df_russell = df_russell[df_russell["Asset Class"] == "Equity"].copy()

tickers_russell = set(df_russell["Ticker"].dropna().unique())

print(f"Empresas en Russell 2500: {len(tickers_russell):,}")
print(f"\nTop 10 sectores:")
print(df_russell["Sector"].value_counts().head(10))
print(f"\nBolsas:")
print(df_russell["Exchange"].value_counts().head(5))

Empresas en Russell 2500: 2,433

Top 10 sectores:
Sector
Financials                574
Health Care               496
Industrials               453
Consumer Discretionary    349
Information Technology    326
Real Estate               171
Materials                 141
Energy                    134
Communication             121
Consumer Staples          109
Name: count, dtype: int64

Bolsas:
Exchange
NASDAQ                              1516
NYSE                                1368
Nyse Mkt Llc                          38
NO MARKET (E.G. UNLISTED)              6
Non-Nms Quotation Service (Nnqs)       2
Name: count, dtype: int64


## 3. Obtención de CIKs para el universo Russell 2500

Para consultar los datos financieros en SEC EDGAR necesitamos el **CIK** (*Central Index Key*) de cada empresa — el identificador único que la SEC asigna a cada entidad registrada.

La SEC publica un archivo con todos los CIKs que cruzamos directamente con los tickers del Russell 2500.

### Filtro sectorial

Para obtener el sector de cada empresa consultamos el endpoint de `submissions` de la SEC, que devuelve el código SIC. Aplicamos el filtro sectorial en este paso para no descargar datos de empresas que no vamos a usar.

> ⚠️ **Esta celda tarda aproximadamente 19 minutos** (solo las ~2.400 empresas del Russell 2500). Solo es necesario ejecutarla una vez — en ejecuciones posteriores carga desde el CSV guardado.

In [7]:
ruta_universo = Path("../datos/crudos/universo_empresas.csv")

if ruta_universo.exists():
    universo = pd.read_csv(ruta_universo)
    print(f"Universo cargado desde archivo: {len(universo):,} empresas")
else:
    # Códigos SIC a excluir
    sic_excluir = [
        range(6000, 6800),  # Financials
        range(4900, 5000),  # Utilities
        range(6500, 6553),  # Real Estate
        range(1300, 1400),  # Oil & Gas
        range(2900, 3000),  # Petroleum
    ]

    def es_excluido(sic):
        try:
            return any(int(sic) in r for r in sic_excluir)
        except:
            return True

    # Descargar archivo de CIKs de la SEC
    url_ciks = "https://www.sec.gov/files/company_tickers_exchange.json"
    datos_ciks = requests.get(url_ciks, headers=headers_sec).json()
    df_ciks = pd.DataFrame(datos_ciks["data"], columns=datos_ciks["fields"])

    # Filtrar solo tickers del Russell 2500 en NYSE/Nasdaq
    df_russell_sec = df_ciks[
        (df_ciks["ticker"].isin(tickers_russell)) &
        (df_ciks["exchange"].isin(["NYSE", "Nasdaq"]))
    ].copy()

    print(f"Tickers Russell 2500 encontrados en SEC: {len(df_russell_sec):,}")

    # Obtener SIC para cada empresa
    info_empresas = []
    errores_sic = []

    for i, fila in df_russell_sec.iterrows():
        cik_fmt = str(fila["cik"]).zfill(10)
        try:
            resp = requests.get(
                f"https://data.sec.gov/submissions/CIK{cik_fmt}.json",
                headers=headers_sec, timeout=10
            )
            if resp.status_code == 200:
                d = resp.json()
                info_empresas.append({
                    "cik":             cik_fmt,
                    "nombre":          d.get("name"),
                    "sic":             d.get("sic"),
                    "sic_descripcion": d.get("sicDescription"),
                    "ticker":          fila["ticker"],
                    "bolsa":           fila["exchange"],
                })
            else:
                errores_sic.append(fila["ticker"])
        except:
            errores_sic.append(fila["ticker"])

        if len(info_empresas) % 200 == 0 and len(info_empresas) > 0:
            print(f"  {len(info_empresas):,} procesadas...")
        time.sleep(0.3)

    df_info = pd.DataFrame(info_empresas)
    df_info["sic"] = pd.to_numeric(df_info["sic"], errors="coerce")

    # Aplicar filtro sectorial
    universo = df_info[~df_info["sic"].apply(es_excluido)].copy().reset_index(drop=True)
    universo.to_csv(ruta_universo, index=False)

    print(f"\nTickers encontrados en SEC:    {len(df_russell_sec):,}")
    print(f"Excluidos por sector:          {len(df_info) - len(universo):,}")
    print(f"Universo final:                {len(universo):,}")
    if errores_sic:
        print(f"Errores al obtener SIC:        {len(errores_sic)}")

print(f"\nEmpresas por bolsa:")
print(universo["bolsa"].value_counts())
print(f"\nTop 10 sectores:")
print(universo["sic_descripcion"].value_counts().head(10))

Tickers Russell 2500 encontrados en SEC: 2,413


  200 procesadas...
  400 procesadas...
  600 procesadas...
  800 procesadas...
  1,000 procesadas...
  1,200 procesadas...
  1,400 procesadas...
  1,600 procesadas...
  1,800 procesadas...
  2,000 procesadas...
  2,200 procesadas...
  2,400 procesadas...

Tickers encontrados en SEC:    2,413
Excluidos por sector:          738
Universo final:                1,675

Empresas por bolsa:
bolsa
Nasdaq    982
NYSE      693
Name: count, dtype: int64

Top 10 sectores:
sic_descripcion
Pharmaceutical Preparations                             183
Services-Prepackaged Software                           100
Biological Products, (No Diagnostic Substances)          66
Surgical & Medical Instruments & Apparatus               54
Semiconductors & Related Devices                         43
Services-Business Services, NEC                          42
Services-Computer Processing & Data Preparation          29
Services-Computer Programming, Data Processing, Etc.     27
Motor Vehicle Parts & Accessories      

## 4. Descarga de variables financieras

Descargamos 40 variables financieras para cada empresa del universo desde SEC EDGAR. Por cada empresa se hace una sola petición HTTP que trae todos sus reportes históricos.

### Estrategia de memoria

Los datos se guardan en disco cada 200 empresas (*chunking*) y se libera la memoria en cada ciclo, evitando que el kernel colapse por uso excesivo de RAM.

> ⚠️ **Esta celda tarda aproximadamente 20 minutos.** Solo es necesario ejecutarla una vez — en ejecuciones posteriores los chunks ya existen.

In [8]:
variables_sec = {
    # Flujo de caja
    "fcf_operativo":       "NetCashProvidedByUsedInOperatingActivities",
    "fcf_inversor":        "NetCashProvidedByUsedInInvestingActivities",
    "fcf_financiero":      "NetCashProvidedByUsedInFinancingActivities",
    "capex":               "PaymentsToAcquirePropertyPlantAndEquipment",
    "depreciacion":        "DepreciationDepletionAndAmortization",
    "stock_based_comp":    "AllocatedShareBasedCompensationExpense",
    "emision_acciones":    "ProceedsFromIssuanceOfCommonStock",
    "recompra_acciones":   "PaymentsForRepurchaseOfCommonStock",
    # Balance general
    "caja":                "CashAndCashEquivalentsAtCarryingValue",
    "activos_totales":     "Assets",
    "activos_corrientes":  "AssetsCurrent",
    "pasivos_corrientes":  "LiabilitiesCurrent",
    "deuda_largo_plazo":   "LongTermDebt",
    "deuda_corto_plazo":   "OtherShortTermBorrowings",
    "inventario":          "InventoryNet",
    "cuentas_por_cobrar":  "AccountsReceivableNetCurrent",
    "cuentas_por_pagar":   "AccountsPayableCurrent",
    "activos_intangibles": "FiniteLivedIntangibleAssetsNet",
    "goodwill":            "Goodwill",
    "ppe_neto":            "PropertyPlantAndEquipmentNet",
    "patrimonio":          "StockholdersEquity",
    "retained_earnings":   "RetainedEarningsAccumulatedDeficit",
    "ingresos_diferidos":  "DeferredIncomeTaxLiabilitiesNet",
    "impuestos_pagados":   "AccruedIncomeTaxesCurrent",
    # Estado de resultados
    "ingresos":            "Revenues",
    "costo_ventas":        "CostOfGoodsAndServicesSold",
    "utilidad_bruta":      "GrossProfit",
    "gastos_id":           "ResearchAndDevelopmentExpense",
    "gastos_operativos":   "OperatingExpenses",
    "gastos_sga":          "SellingGeneralAndAdministrativeExpense",
    "utilidad_operativa":  "OperatingIncomeLoss",
    "utilidad_neta":       "NetIncomeLoss",
    "ebitda_proxy":        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
    "intereses_pagados":   "InterestPaid",
    "amortizacion_intang": "AmortizationOfIntangibleAssets",
    # Capital de trabajo
    "acciones_circulacion":"CommonStockSharesOutstanding",
    "cambio_inventario":   "IncreaseDecreaseInInventories",
    "cambio_cxc":          "IncreaseDecreaseInAccountsReceivable",
    "otras_inversiones":   "GoodwillAcquiredDuringPeriod",
    "cambio_capital_trab": "IncreaseDecreaseInOtherOperatingAssets",
}

print(f"Variables a descargar: {len(variables_sec)}")

Variables a descargar: 40


In [9]:
def extraer_variable(us_gaap, nombre, concepto, ticker, sic_desc, bolsa):
    """Extrae una variable financiera del JSON de la SEC para una empresa."""
    if concepto not in us_gaap:
        return None
    registros = us_gaap[concepto]["units"].get("USD", [])
    if not registros:
        registros = us_gaap[concepto]["units"].get("shares", [])
    if not registros:
        return None
    df = pd.DataFrame(registros)
    df = df[df["form"].isin(["10-Q", "10-K"])].copy()
    if df.empty:
        return None
    df["ticker"]   = ticker
    df["sector"]   = sic_desc
    df["bolsa"]    = bolsa
    df["variable"] = nombre
    return df


chunks_path = Path("../datos/crudos/chunks")
chunks_existentes = sorted(chunks_path.glob("chunk_*.csv"))

if chunks_existentes:
    print(f"Chunks ya existentes: {len(chunks_existentes)} — saltando descarga.")
    print("Procede directamente a la celda de consolidación.")
else:
    lote           = []
    empresas_ok    = []
    empresas_error = []
    num_chunk      = 0
    GUARDAR_CADA   = 200

    for i, fila in universo.iterrows():
        ticker   = fila["ticker"]
        cik      = str(fila["cik"]).zfill(10)
        sic_desc = fila["sic_descripcion"]
        bolsa    = fila["bolsa"]

        try:
            respuesta = requests.get(
                f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json",
                headers=headers_sec, timeout=30
            )
            if respuesta.status_code != 200:
                empresas_error.append({
                    "ticker": ticker, "cik": cik,
                    "bolsa": bolsa, "sector": sic_desc,
                    "motivo": f"HTTP {respuesta.status_code}"
                })
                continue

            us_gaap = respuesta.json().get("facts", {}).get("us-gaap", {})
            vars_encontradas = 0

            for nombre, concepto in variables_sec.items():
                df_var = extraer_variable(us_gaap, nombre, concepto, ticker, sic_desc, bolsa)
                if df_var is not None:
                    lote.append(df_var)
                    vars_encontradas += 1

            empresas_ok.append({
                "ticker": ticker, "cik": cik,
                "bolsa": bolsa, "sector": sic_desc,
                "variables_descargadas": vars_encontradas
            })

        except Exception as e:
            empresas_error.append({
                "ticker": ticker, "cik": cik,
                "bolsa": bolsa, "sector": sic_desc,
                "motivo": str(e)[:100]
            })

        total = len(empresas_ok) + len(empresas_error)
        if total % GUARDAR_CADA == 0 and lote:
            pd.concat(lote, ignore_index=True).to_csv(
                f"../datos/crudos/chunks/chunk_{num_chunk:04d}.csv", index=False
            )
            num_chunk += 1
            lote = []
            print(f"  {total:,}/{len(universo):,} — chunk {num_chunk} guardado — "
                  f"{len(empresas_ok):,} ok, {len(empresas_error):,} errores")
        time.sleep(0.3)

    # Último lote
    if lote:
        pd.concat(lote, ignore_index=True).to_csv(
            f"../datos/crudos/chunks/chunk_{num_chunk:04d}.csv", index=False
        )
        num_chunk += 1

    pd.DataFrame(empresas_ok).to_csv("../datos/crudos/empresas_exitosas.csv", index=False)
    pd.DataFrame(empresas_error).to_csv("../datos/crudos/empresas_con_error.csv", index=False)

    print(f"\nDescarga completa.")
    print(f"  Empresas exitosas:  {len(empresas_ok):,}")
    print(f"  Empresas con error: {len(empresas_error):,}")
    print(f"  Chunks guardados:   {num_chunk}")

  200/1,675 — chunk 1 guardado — 200 ok, 0 errores
  400/1,675 — chunk 2 guardado — 400 ok, 0 errores
  600/1,675 — chunk 3 guardado — 600 ok, 0 errores
  800/1,675 — chunk 4 guardado — 800 ok, 0 errores
  1,000/1,675 — chunk 5 guardado — 1,000 ok, 0 errores
  1,200/1,675 — chunk 6 guardado — 1,200 ok, 0 errores
  1,400/1,675 — chunk 7 guardado — 1,399 ok, 1 errores
  1,600/1,675 — chunk 8 guardado — 1,599 ok, 1 errores

Descarga completa.
  Empresas exitosas:  1,674
  Empresas con error: 1
  Chunks guardados:   9


## 5. Consolidación y guardado

### 5.1 Consolidar chunks

Los chunks se combinan en un solo archivo comprimido en formato `.csv.gz` (gzip), que reduce el tamaño ~94% manteniendo compatibilidad total con pandas.

In [10]:
archivos = sorted(Path("../datos/crudos/chunks").glob("chunk_*.csv"))
print(f"Chunks encontrados: {len(archivos)}")

df_variables = pd.concat(
    [pd.read_csv(f) for f in archivos],
    ignore_index=True
)

# Guardar comprimido
ruta_gz = Path("../datos/crudos/variables_financieras_sec.csv.gz")
df_variables.to_csv(ruta_gz, compression="gzip", index=False)

# Eliminar chunks
shutil.rmtree("../datos/crudos/chunks")

tamano_mb = ruta_gz.stat().st_size / (1024 * 1024)
print(f"\nDataset guardado: variables_financieras_sec.csv.gz")
print(f"Tamaño comprimido: {tamano_mb:.1f} MB")
print(f"Shape: {df_variables.shape}")
print(f"Empresas: {df_variables['ticker'].nunique():,}")
print(f"Variables: {df_variables['variable'].nunique()}")
print(f"\nFormularios:")
print(df_variables["form"].value_counts())

Chunks encontrados: 9

Dataset guardado: variables_financieras_sec.csv.gz
Tamaño comprimido: 36.0 MB
Shape: (4472817, 13)
Empresas: 1,626
Variables: 40

Formularios:
form
10-Q    3135424
10-K    1337393
Name: count, dtype: int64


### 5.2 Documentar empresas sin datos US GAAP

Algunas empresas del Russell 2500 no tienen datos bajo el estándar US GAAP en SEC EDGAR. Corresponden principalmente a ADRs de empresas extranjeras. Se documentan para la metodología de la tesis.

In [11]:
df_ok = pd.read_csv("../datos/crudos/empresas_exitosas.csv")
tickers_con_datos = set(df_variables["ticker"].unique())

sin_datos = df_ok[~df_ok["ticker"].isin(tickers_con_datos)].copy()
sin_datos["motivo_exclusion"] = "Sin datos US GAAP en SEC EDGAR (ADR o empresa extranjera)"
sin_datos.to_csv("../datos/crudos/empresas_sin_datos_usgaap.csv", index=False)

print(f"Empresas descargadas exitosamente: {len(df_ok):,}")
print(f"Con datos US GAAP:                 {len(tickers_con_datos):,}")
print(f"Sin datos US GAAP (ADRs):          {len(sin_datos):,}")
if len(sin_datos) > 0:
    print(f"\nEjemplos:")
    print(sin_datos[["ticker", "bolsa", "sector"]].head(8).to_string(index=False))

Empresas descargadas exitosamente: 1,674
Con datos US GAAP:                 1,626
Sin datos US GAAP (ADRs):          48

Ejemplos:
ticker  bolsa                                                   sector
    AS   NYSE Apparel & Other Finishd Prods of  Fabrics & Similar Matl
  TIGO Nasdaq                            Radiotelephone Communications
   JHX   NYSE                  Concrete Products, Except Block & Brick
  QGEN   NYSE          Biological Products, (No Diagnostic Substances)
   DOX Nasdaq                   Services-Computer Programming Services
  BIRK   NYSE                                    Footwear, (No Rubber)
  SGHC   NYSE            Services-Miscellaneous Amusement & Recreation
  GLNG Nasdaq                                     Water Transportation


## 6. Resumen final

Verificación completa del dataset antes de pasar al Cuaderno 2.

In [12]:
print("=" * 55)
print("RESUMEN METODOLÓGICO")
print("=" * 55)
print(f"  Constituyentes Russell 2500 (SMMD):    {len(tickers_russell):,}")
print(f"  Encontrados en SEC EDGAR:              {len(universo) + (len(df_info) - len(universo)):,}")
print(f"  Excluidos por sector:                  {len(df_info) - len(universo):,}")
print(f"  Universo final (post-filtro):          {len(universo):,}")
print(f"  Sin datos US GAAP (ADRs):              {len(sin_datos):,}")
print(f"  Dataset final:                         {df_variables['ticker'].nunique():,}")
print()
print("=" * 55)
print("DATASET FINAL")
print("=" * 55)
print(f"  Registros totales:   {len(df_variables):,}")
print(f"  Empresas:            {df_variables['ticker'].nunique():,}")
print(f"  Variables:           {df_variables['variable'].nunique()}")
print(f"  Formularios 10-Q:    {(df_variables['form']=='10-Q').sum():,}")
print(f"  Formularios 10-K:    {(df_variables['form']=='10-K').sum():,}")
print(f"  Desde:               {df_variables['end'].min()}")
print(f"  Hasta:               {df_variables['end'].max()}")
print()
print("EMPRESAS POR BOLSA")
print(df_variables.groupby("bolsa")["ticker"].nunique())
print()
print("TOP 10 SECTORES")
print(df_variables.groupby("sector")["ticker"].nunique().sort_values(ascending=False).head(10))

print()
print("ARCHIVOS GENERADOS EN datos/crudos:")
for f in sorted(Path("../datos/crudos").iterdir()):
    tam = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {tam:.1f} MB")

RESUMEN METODOLÓGICO
  Constituyentes Russell 2500 (SMMD):    2,433
  Encontrados en SEC EDGAR:              2,413
  Excluidos por sector:                  738
  Universo final (post-filtro):          1,675
  Sin datos US GAAP (ADRs):              48
  Dataset final:                         1,626

DATASET FINAL
  Registros totales:   4,472,817
  Empresas:            1,626
  Variables:           40
  Formularios 10-Q:    3,135,424
  Formularios 10-K:    1,337,393
  Desde:               1980-07-21
  Hasta:               2028-12-28

EMPRESAS POR BOLSA
bolsa
NYSE      662
Nasdaq    964
Name: ticker, dtype: int64

TOP 10 SECTORES
sector
Pharmaceutical Preparations                             180
Services-Prepackaged Software                            99
Biological Products, (No Diagnostic Substances)          65
Surgical & Medical Instruments & Apparatus               54
Semiconductors & Related Devices                         43
Services-Business Services, NEC                          41
